# Instacart Market Basket Analysis

This notebook analyzes Instacart order data to understand customer ordering behavior, basket composition, and reorder patterns. The workflow reflects a complete exploratory analysis process: inspecting structure, cleaning inconsistencies, combining related tables, and translating results into business-relevant observations.

## Project setup

I begin by importing the libraries I use for data preparation, analysis, and visualization.

In [ ]:
# Import the libraries you'll need for this analysis
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Import the libraries used throughout the analysis
import pandas as pd
import matplotlib.pyplot as plt

# Load the Kaggle Instacart files stored in the local datasets/ folder
base_path = 'datasets/'

orders = pd.read_csv(base_path + 'orders.csv')
products = pd.read_csv(base_path + 'products.csv')
departments = pd.read_csv(base_path + 'departments.csv')
aisles = pd.read_csv(base_path + 'aisles.csv')

order_products_prior = pd.read_csv(base_path + 'order_products__prior.csv')
order_products_train = pd.read_csv(base_path + 'order_products__train.csv')

order_products = pd.concat(
    [order_products_prior, order_products_train],
    ignore_index=True
)


## Initial review of the data

I start by reviewing each dataset with `.head()` and `.info()` so I can understand its structure, data types, and completeness before making any cleaning decisions.

In [ ]:
# In this cell, type "orders" below this line and execute the cell
print(orders.head())

In [ ]:
# In this cell, type "products" below this line and execute the cell
print(products.head())

I repeat this review across all tables to see how the datasets relate to one another and to identify any issues that may affect later analysis.

In [ ]:
# In this cell, type "orders.info() below this line and execute the cell
print(orders.info(show_counts=True))

As I review the outputs, I focus on the non-null counts to identify columns with missing data and decide where closer investigation is needed.

In [ ]:
# In this cell, run orders_products.info() below, but include the argument show_counts=True since this is a large file.
print(order_products.info(show_counts=True))

I apply the same structure check to the remaining tables so I can spot missing values and confirm that each dataset is ready for the next step.

In [ ]:
# Continue the structure review for the remaining lookup tables
print('-' * 84)
print('Products info')
print(products.info(show_counts=True))

print('-' * 84)
print('Departments info')
print(departments.info(show_counts=True))

print('-' * 84)
print('Aisles info')
print(aisles.info(show_counts=True))


## Missing-value review and treatment

After the initial inspection, I investigate each column with missing values and decide how to handle it based on context rather than applying a single blanket rule.

### Products table

I begin with the `products` table because missing product names would make later product-level analysis harder to interpret.

In [ ]:
# Display rows where the product_name column has missing values
print(products[products['product_name'].isna()])

The missing `product_name` values appear concentrated in rows associated with `aisle_id` 100 and `department_id` 21. I test that pattern directly to confirm whether it is isolated or part of a broader data issue.

In [ ]:
# Combine conditions to check for missing product names in aisles other than 100
oa_filter = (products['product_name'].isna()) & (products['aisle_id'] != 100)
other_aisles = products[oa_filter]
print(other_aisles)

In [ ]:
# Combine conditions to check for missing product names in departments other than 21
d_filter = (products['product_name'].isna()) & (products['department_id'] != 21)
other_departments = products[d_filter]
print(other_departments)

To understand those rows more clearly, I look up the related aisle and department labels in the supporting tables.

In [ ]:
# What is this aisle and department?
print(departments[departments['department_id'] == 21])
print(aisles[aisles['aisle_id'] == 100])

In [ ]:
# Fill missing product names with 'Unknown'
products['product_name'] = products['product_name'].fillna('Unknown')

### Orders table

I then review missing values in the `orders` table to determine whether they reflect a data-quality problem or a normal feature of customer behavior.

In [ ]:
# Display rows where the days_since_prior_order column has missing values
print(orders[orders['days_since_prior_order'].isna()])

In [ ]:
# Are there any missing values where it's not a customer's first order?
print(orders[(orders['days_since_prior_order'].isna()) & (orders['order_number'] != 1)])

All of the missing `'days_since_prior_order'` values correspond to a customer's first ever order. This makes sense because there is no prior order! We'll leave the values as `NaN` so the column can remain numeric. Also, the `NaN` values shouldn't interfere with any calculations we might do using this column.

### Order-products table

I next examine missing values in `order_products`, where the issue affects the recorded sequence in which items were added to the cart.

In [ ]:
# Display rows where the add_to_cart_order column has missing values
print(order_products[order_products['add_to_cart_order'].isna()])

In [ ]:
# Use .min() and .max() to find the minimum and maximum values for this column.
print(f"The minimum value for the add_to_cart_order column is {int(order_products['add_to_cart_order'].min())}.")
print()
print(f"The maximum value for this column is {int(order_products['add_to_cart_order'].max())}.")

In [ ]:
# Save all order IDs with at least one missing value in 'add_to_cart_order'
import numpy as np #Importing Numerical Python to use tools within.
orders_with_missing = np.unique(order_products[order_products['add_to_cart_order'].isna()]['order_id']) #Saving Order ID's with missing Add-To-Cart data to new variable.
print(f"There are {len(orders_with_missing)} order IDs with at least one missing value in the 'add_to_cart_order' column.") #Counts the number of orders with at least one missing value in 'add_to_cart_order'

In [ ]:
# Do all orders with missing values have more than 64 products?
filter_64 = ((order_products['order_id'].isin(orders_with_missing)) & (order_products['add_to_cart_order'] == 64))
orders_with_64th_item = order_products[filter_64]
print(orders_with_64th_item) #All 70 order IDs have at least 64 products added to the cart, meaning an error happend with the 'add_to_cart_order' values after 64. 

In [ ]:
# Replace missing values with 999 and convert column to integer type
order_products['add_to_cart_order'] = order_products['add_to_cart_order'].fillna(999)

The missing values in `add_to_cart_order` appear only for items placed 65th or later in the cart. I preserve those rows by assigning a placeholder value and keep that decision in mind when interpreting basket-position results later.

## Duplicate-value review

With missing values addressed, I check each dataset for duplicate records so the analysis is built on cleaner and more reliable data.

### `orders` data frame

In [ ]:
# Find the number of duplicate rows in the orders dataframe
print(orders.duplicated().sum())

In [ ]:
# View the duplicate rows
print(orders[orders.duplicated()])

In [ ]:
# Remove duplicate orders
orders = orders.drop_duplicates().reset_index(drop=True)

In [ ]:
# Double check for duplicate rows
print(orders.duplicated().sum())

### `products` data frame

In [ ]:
# Check for fully duplicate rows
print(products[products.duplicated()])

In [ ]:
# Check for just duplicate product IDs using subset='product_id' in duplicated()
print(products[products['product_id'].duplicated()])

To evaluate duplicate product names consistently, I convert names to lowercase so capitalization differences do not hide genuine duplicates.

In [ ]:
# Check for just duplicate product names (convert names to lowercase to compare better)
products['product_name'] = products['product_name'].str.lower()
print(products[products['product_name'].duplicated()])

I then inspect how those duplicate product names appear in the data so I can distinguish between repeated names and truly duplicated records.

In [ ]:
products[products['product_name'].str.lower() == 'high performance energy drink']

In [ ]:
# Drop duplicate product names (case insensitive)
products['product_name'].drop_duplicates().reset_index(drop = True)

### `departments` data frame

In [ ]:
# Check for duplicate entries in the departments dataframe
print(departments[departments.duplicated()])

### `aisles` data frame

In [ ]:
# Check for duplicate entries in the aisles dataframe
print(aisles[aisles.duplicated()])

### `order_products` data frame

In [ ]:
# Check for duplicate entries in the order_products dataframe
print(order_products[order_products.duplicated()])

With the main cleaning steps complete, I move into exploratory analysis to understand order timing, basket size, product popularity, and reorder behavior.

### Validating time fields

Before analyzing order patterns, I verify that `order_hour_of_day` and `order_dow` fall within sensible ranges.

In [ ]:
print('Orders hour of the day include:')
print(orders['order_hour_of_day'].unique())
print()
print('Orders day of the week include:')
print(orders['order_dow'].unique())

In [ ]:
print('Orders hour of the day (sorted):')
print(sorted(orders['order_hour_of_day'].unique()))
print()
print('Orders day of the week (sorted):')
print(sorted(orders['order_dow'].unique()))

### Shopping patterns by hour

I analyze `order_hour_of_day` to identify when customers are most active and to highlight the daily rhythm of grocery shopping.

In [ ]:
from matplotlib import pyplot as plt

hourly_orders = orders['order_hour_of_day'].value_counts().sort_index()

hourly_orders.plot(title = 'Time of Day People Shop for Groceries',
                  xlabel = 'Hour in the Day',
                  ylabel = 'Volume of Orders',
                  style = 'o-',
                  grid = True,
                   xticks = range(0, 23, 1)
                  )

plt.show()

#### Analysis:
- The busiest time of day is between 8 AM and 6 PM.
- Orders peak at 10 AM, making it a critical time of day for ensuring Instacart delivery drivers and Instacart servers are ready for the order volume.

Most orders occur between 9:00 AM and 5:00 PM, with peaks at 10:00 AM and 3:00 PM

### Shopping patterns by day of week

I analyze `order_dow` to compare order volume across the week and see whether activity clusters around particular days.

In [ ]:
orders_per_day = orders['order_dow'].value_counts().sort_index()

orders_per_day.plot(title = 'Days People Shop for Groceries',
                   ylabel = 'Volume of Orders',
                   xlabel = 'Day of the Week',
                   style = 'o-',
                    grid = True,
                    xticks = range(0, 6, 1)
                   )

plt.show()

#### Analysis:
- Assuming Sunday = 0, customers place most of their orders on Sunday and Monday.
- Orders sharply drop off on Tuesday, Wednesday, and Thursday.
- On Friday, orders rise a bit and then drop slightly on Saturday. This seems to indicate people are either finishing up the work week or getting some supplies for their weekend plans.

The data dictionary does not state which integer corresponds to which day of the week. Assuming Sunday = 0, then people place more orders at the beginning of the week (Sunday and Monday).

### Time between orders

I examine `days_since_prior_order` to understand how frequently customers return to place another order.

In [ ]:
days_until_reordering = orders['days_since_prior_order'].value_counts().sort_index()

days_until_reordering.plot(title = 'Days Until Placing Another Order',
                          ylabel = 'Order Volume',
                          xlabel = 'Days Elapsed Since Last Order',
                          grid = True,
                          xticks = range(0, 30, 2),
                          rot = 45)

plt.show()

#### Analysis:
- Most orders are made 7 days after the last order.
- This is a great data point for understanding the most common frequency customers like to reorder at.
- There are other spikes in orders at the 2-week, 3-week, and 4-week mark.
- This is likely due to having automatically reoccuring subscriptions set to place an order ever 1, 2, 3, or 4 weeks.

The 0 values probably correspond to customers who placed more than one order on the same day.

The max value of 30 days and the high spike at that value is puzzling though. The spike might be explained by people who set up recurring subscriptions to automatically order once a month. But that doesn't explain why there are no values above 30 days. I would expect many customers to place orders less often than once a month. Maybe those customers were intentionally excluded from the dataset.

Disregarding the spike at 30 days, most people wait between 2 to 10 days in between orders. The most common wait time is 7 days. In other words, it's common for people to place weekly grocery orders. Interestingly, in the tail of the distribution we also see small spikes at 14, 21, and 28 days. These would correspond to orders every 2, 3, or 4 weeks.

### Comparing Wednesday and Saturday order patterns

I compare the hourly order distributions for Wednesday and Saturday to see whether customer behavior differs between a weekday and a weekend day.

In [ ]:
wednesday_orders = orders[orders['order_dow'] == 3]['order_hour_of_day'].value_counts().sort_index()
saturday_orders = orders[orders['order_dow'] == 6]['order_hour_of_day'].value_counts().sort_index()

In [ ]:
wednesday_and_saturday = pd.concat([wednesday_orders, saturday_orders], axis = 'columns')

In [ ]:
wednesday_and_saturday.columns = ['wednesday_orders', 'saturday_orders']

In [ ]:
wednesday_and_saturday.plot(title = 'Orders on Wednesdays and Saturdays',
                           xlabel = 'Hour of the Day',
                           ylabel = 'Volume of Orders',
                           grid = True,
                           xticks = range(0, 24, 1),
                            yticks = range(0, 5500, 500),
                           kind = 'bar')
plt.show()

#### Analysis:
- Wednesday does experience a dip in orders midday, relative to Saturday. This is likely due to work obligations or because they're currently focused on getting lunch.
- The most popular time, by far, is between 9 am and 5 pm.
- It's interesting how orders between 12 am and 5 am are negligable, while orders stay relatively high up until 11pm. It looks like a decent segment of customers might be having some late-night cravings or doing some last-minute shopping.

There's a small dip from 11h to 13h on Wednesdays. This dip is absent on Saturdays. Maybe this dip can be attributed to people who don't use Instacart because they have lunch somewhere between 11h and 13h.

### Orders per customer

I examine how many orders each customer has placed to understand how concentrated repeat activity is across the user base.

In [ ]:
orders_per_customer = orders.groupby('user_id')['order_id'].count().sort_values()

In [ ]:
orders_per_customer.plot(kind = 'hist',
                        bins = 16,
                        xticks = range(1, 17, 1),
                        range = (0, 16),
                        yticks = range(0, 57000, 4000),
                        title = 'Orders Per Customer',
                        grid = True,
                        ylim = (0, 57000))

plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')

plt.show()

#### Analysis:
- The large majority of customers in the dataset have only placed between 1 and 6 orders.
- This data could be used to target promotions at customers most likely to reorder. Typically, those are the customers who have already reordered. Efforts could be focused from most loyal to least loyal customer, making promotions more efficient.

Most customers in the dataset have placed between 1 and 10 orders, with number of orders per customer sharply decreasing after just 1 order.

### Most popular products

To identify the most frequently purchased items, I merge product metadata into the order data and rank products by total purchase count.

In [ ]:
orders_and_products = order_products.merge(products,
                                          left_on = 'product_id',
                                          right_on = 'product_id')

In [ ]:
grouped_products = orders_and_products.groupby(['product_id', 'product_name']).size()
grouped_products = grouped_products.sort_values(ascending=False)
top_20 = grouped_products.head(20)
print(top_20)

In [ ]:
top_20.plot(title = 'Top 20 Popular Products', 
            ylabel = 'Number of Orders', 
            xlabel = 'Products',
            figsize = (12, 6),
            kind = 'bar')

plt.show()

#### Analysis
- The faster a product is likely to **spoil**, the more frequently it is reordered.
- Milk is the one item that stands out as not being in the produce department.

The top 20 items are all produce, except for the milk. Looks like people want delicious and nutritious!

### Items per order

I calculate how many products appear in each order to understand typical basket size and the shape of the overall distribution.

In [ ]:
group_ops = order_products.groupby('order_id')
number_per_order = group_ops['product_id'].count()

In [ ]:
items_per_order = number_per_order.value_counts().sort_index()

#Added (Before Filtering) to make note that this is not the final visualization. The next will focus on the majority of the data.
items_per_order.plot(kind = 'bar',
                    title = 'Items Per Order (Before Filtering)',
                    xticks = range(0, 50, 5),
                    xlabel = 'Number of Items',
                    ylabel = 'Volume of Orders')

plt.show()


Most of the order numbers are in the tail of the distribution. To get a better look at the non-tail part, let's choose a value in the tail as a cutoff and just plot order with fewer than that many items. An order size of 35 items is far enough into the tail for this.

In [ ]:
filtered_ipo = items_per_order[items_per_order.index <= 35] #Creating a filtered version of 'items_per_order'
filtered_ipo.plot(kind = 'bar',
                  title = 'Items Per Order',
                  xlabel = 'Number of Items',
                  ylabel = 'Volume of Orders')

plt.show()

#### Analysis:
- Most orders include a total of 5 items, with 6 and 4 being the next most frequent number of items in a customer's cart before checkout.
- This data can be useful for limiting the scope of how many items are recommended at any given time, allowing them to quickly picture their order without much navigating.

The typical order contains 5 or 6 items, with most orders having between 1 and 20 items.

### Most frequently reordered products

I isolate rows where `reordered == 1`, merge in product names, and rank products by how often they return to customers' carts.

In [ ]:
reordered_products = order_products[order_products['reordered'] == 1] #Filtering the DataFrame to only reordered products

In [ ]:
#Combining 'reordered_products' and 'products' DataFrames on 'product_id'. This variable 'rp_combo' stands for reordered_products combination.
rp_combo = reordered_products.merge(products,
                                   left_on = 'product_id',
                                   right_on = 'product_id')

rp_combo = rp_combo.groupby(['product_id', 'product_name']).size() #Calculating how many times each product was reordered.

In [ ]:
rp_combo = rp_combo.sort_values(ascending = False) #Sorting values in descending order.

rp_top_20 = rp_combo.head(20) #Filtering down to the top 20 most reordered items and storing them into a 'rp_top_20' variable.

In [ ]:
 

rp_top_20.plot(title = 'Top 20 Most Reordered Items',
              kind = 'bar',
              xlabel = 'Items',
              ylabel = 'Frequency')

plt.show()

#### Analysis
- The first three most reordered items really stand out: bananas, bag of organic bananas, and organic strawberies.
- After the first five most frequently reordered items, the frequency really starts to drop off.
- This information could be used for recommendation engines, especially when trying to recommend something to a new customer no data yet exists for.

It looks like produce and dairy comprise the most reordered products as well. It makes sense that perishables would be the most reordered items.

### Reorder proportion by product

I then shift from raw reorder counts to reorder rates by measuring what share of each product's orders are reorders rather than first-time purchases.

In [ ]:
# I used the variable 'rp_per_item' to stand for 'Reorder Proportions Per Item', since this is what we are aiming to narrow in on.
rp_per_item = order_products.merge(products) #Merged 'order_products' and 'products' datasets.
rp_per_item = rp_per_item.groupby(['product_id', 'product_name']) #Isolated each product's order history
reorder_proportions = rp_per_item['reordered'].mean() #Here, I calculated teh reorder proportions per item.
sorted_rps = reorder_proportions.sort_values(ascending = False) #I used the new variable 'sorted_rps' to stand for 'Sorted Reorder Proportions'.
sorted_rps.reset_index() #Organized data into a readable DataFrame

In [ ]:
# Sample products from reorder-rate ranges to see what types of items appear at different levels
for low in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0]:
    high = low + 0.1
    if low == 0.9:
        subset = sorted_rps[(sorted_rps >= low) & (sorted_rps <= 1.0)]
    else:
        subset = sorted_rps[(sorted_rps >= low) & (sorted_rps < high)]

    print(f"\nReorder-rate range {low:.1f} to {high:.1f}:")
    if len(subset) == 0:
        print("No products in this range.")
    else:
        print(subset.sample(min(10, len(subset)), random_state=42))


#### Analysis:
- Reorder frequency appears to track how quickly a product is consumed or replaced. 
- The highest-reordered items are mostly perishable, everyday consumables, while the lowest-reordered items are typically durable, occasional-use, or one-time purchases.
- These reorder rates are useful for both recommendation systems, where it can guide repurchase suggestions, and forecasting, where it can help anticipate restocking demand.